# 03 - Phân tích Oxford-IIIT Pet và giao thức CNN

Notebook này chuẩn bị workflow phân loại Oxford-IIIT Pet cho Assignment 05. Notebook chỉ dùng các mẫu annotation chính thức, giữ official test split không bị chạm tới, và chỉ tuning BasicCNN2D để chọn giao thức huấn luyện cấp dataset.


## 1. Định nghĩa bài toán

Bài toán là phân loại giống thú cưng từ ảnh RGB với 37 lớp. Assignment sẽ so sánh các family kiến trúc CNN ở phần sau, nhưng notebook này chỉ tạo split hợp lệ và chọn protocol bằng validation evidence.


In [ ]:
from pathlib import Path
import json
import random
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from PIL import Image
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

from models.architectures import build_basic_cnn_2d

SEED = 42
EXPECTED_PYTHON = "C:/Users/anhca/anaconda3/envs/tf312/python.exe"
actual_python = sys.executable.replace("\\", "/")
assert actual_python.lower() == EXPECTED_PYTHON.lower(), actual_python

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "AGENTS.md").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("A05 project root not found")
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "datasets" / "oxford_pets"
IMAGE_DIR = DATA_DIR / "images"
ANNOTATION_DIR = DATA_DIR / "annotations"
SPLIT_DIR = PROJECT_ROOT / "results" / "splits"
HP_DIR = PROJECT_ROOT / "results" / "hyperparameters"
FIG_ROOT = PROJECT_ROOT / "results" / "figures" / "oxford_pets"
HP_FIG_DIR = FIG_ROOT / "hyperparameters"
EXAMPLE_DIR = FIG_ROOT / "examples"
for path in [SPLIT_DIR, HP_DIR, HP_FIG_DIR, EXAMPLE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

required_paths = [
    IMAGE_DIR,
    ANNOTATION_DIR / "trainval.txt",
    ANNOTATION_DIR / "test.txt",
    ANNOTATION_DIR / "list.txt",
]
missing = [
    path.relative_to(PROJECT_ROOT).as_posix()
    for path in required_paths
    if not path.exists()
]
if missing:
    raise FileNotFoundError(f"Required Oxford Pets files are missing: {missing}")

print(
    {
        "python_executable": actual_python,
        "python_version": sys.version.split()[0],
        "tensorflow_version": tf.__version__,
        "keras_version": keras.__version__,
        "devices": [str(device) for device in tf.config.list_physical_devices()],
        "project_root": PROJECT_ROOT.name,
    }
)

{'python_executable': 'C:/Users/anhca/anaconda3/envs/tf312/python.exe', 'python_version': '3.12.14', 'tensorflow_version': '2.21.0', 'keras_version': '3.15.1', 'devices': ["PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')"], 'project_root': 'A05'}


## 2. Mô tả dataset

Nguồn dữ liệu cục bộ là `datasets/oxford_pets/`. Raw images là read-only. Thư mục ảnh có thêm một số raw file, nhưng mẫu phân loại chỉ được lấy từ annotation chính thức.


## 3. Cấu trúc annotation chính thức

`annotations/trainval.txt` và `annotations/test.txt` là nguồn canonical. Mỗi dòng cung cấp image id, class id, species id, và breed id. Thí nghiệm không glob toàn bộ thư mục `images/` để tạo sample.


In [ ]:
def class_name_from_image_id(image_id):
    return image_id.rsplit("_", 1)[0]


def parse_annotation_file(path, split_name):
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            image_id, class_id, species, breed_id = stripped.split()
            image_path = IMAGE_DIR / f"{image_id}.jpg"
            records.append(
                {
                    "image_id": image_id,
                    "relative_path": image_path.relative_to(PROJECT_ROOT).as_posix(),
                    "class_id": int(class_id),
                    "label": int(class_id) - 1,
                    "class_name": class_name_from_image_id(image_id),
                    "species": int(species),
                    "breed_id": int(breed_id),
                    "official_split": split_name,
                }
            )
    return pd.DataFrame(records)


official_trainval_df = parse_annotation_file(
    ANNOTATION_DIR / "trainval.txt", "trainval"
)
official_test_df = parse_annotation_file(ANNOTATION_DIR / "test.txt", "test")
official_manifest = pd.concat(
    [official_trainval_df, official_test_df], ignore_index=True
)

trainval_ids = set(official_trainval_df["image_id"])
test_ids = set(official_test_df["image_id"])
assert len(official_trainval_df) == 3680
assert len(official_test_df) == 3669
assert len(official_manifest["image_id"].unique()) == 7349
assert official_manifest["class_id"].nunique() == 37
assert len(trainval_ids & test_ids) == 0

png_content_jpg = {
    "Abyssinian_5",
    "Egyptian_Mau_14",
    "Egyptian_Mau_156",
    "Egyptian_Mau_186",
}
referenced_png_mismatch = sorted(png_content_jpg & set(official_manifest["image_id"]))

print(
    {
        "trainval": len(official_trainval_df),
        "test": len(official_test_df),
        "official_total": len(official_manifest),
        "classes": official_manifest["class_id"].nunique(),
        "trainval_test_overlap": len(trainval_ids & test_ids),
        "png_content_jpg_referenced": referenced_png_mismatch,
    }
)
official_manifest.head()

{'trainval': 3680, 'test': 3669, 'official_total': 7349, 'classes': 37, 'trainval_test_overlap': 0, 'png_content_jpg_referenced': ['Abyssinian_5', 'Egyptian_Mau_14', 'Egyptian_Mau_156', 'Egyptian_Mau_186']}


         image_id  ... official_split
0  Abyssinian_100  ...       trainval
1  Abyssinian_101  ...       trainval
2  Abyssinian_102  ...       trainval
3  Abyssinian_103  ...       trainval
4  Abyssinian_104  ...       trainval

[5 rows x 8 columns]

In [ ]:
image_records = []
for row in official_manifest.itertuples(index=False):
    image_path = PROJECT_ROOT / row.relative_path
    with Image.open(image_path) as image:
        detected_format = image.format
        rgb = image.convert("RGB")
        width, height = rgb.size
    image_records.append(
        {
            "image_id": row.image_id,
            "width": width,
            "height": height,
            "channels": 3,
            "detected_format": detected_format,
            "extension": image_path.suffix.lower(),
            "format_mismatch": image_path.suffix.lower() == ".jpg"
            and detected_format != "JPEG",
        }
    )

image_info_df = pd.DataFrame(image_records)
official_manifest = official_manifest.merge(image_info_df, on="image_id", how="left")
raw_jpg_ids = {path.stem for path in IMAGE_DIR.glob("*.jpg")}
unreferenced_raw_ids = sorted(raw_jpg_ids - set(official_manifest["image_id"]))

integrity_summary = {
    "official_samples": int(len(official_manifest)),
    "decoded_samples": int(official_manifest["width"].notna().sum()),
    "format_counts": official_manifest["detected_format"].value_counts().to_dict(),
    "format_mismatch_count": int(official_manifest["format_mismatch"].sum()),
    "unreferenced_raw_images": len(unreferenced_raw_ids),
}
print(json.dumps(integrity_summary, indent=2))
official_manifest[official_manifest["format_mismatch"]][
    ["image_id", "relative_path", "detected_format", "width", "height"]
]

{
  "official_samples": 7349,
  "decoded_samples": 7349,
  "format_counts": {
    "JPEG": 7345,
    "PNG": 4
  },
  "format_mismatch_count": 4,
  "unreferenced_raw_images": 41
}


              image_id  ... height
2395   Egyptian_Mau_14  ...    800
2402  Egyptian_Mau_156  ...    265
2427  Egyptian_Mau_186  ...    275
3735      Abyssinian_5  ...    150

[4 rows x 5 columns]

## 4. Phân bố lớp

Phân bố được kiểm tra riêng cho official trainval và official test. Validation split chỉ được tạo từ trainval, dùng stratification để giữ tỷ lệ lớp gần nhất có thể.


In [ ]:
class_distribution = (
    official_manifest.groupby(["class_id", "class_name", "official_split"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
class_distribution["total"] = class_distribution.get(
    "trainval", 0
) + class_distribution.get("test", 0)
class_distribution.head(12)

official_split  class_id                 class_name  test  trainval  total
0                      1                 Abyssinian    98       100    198
1                      2           american_bulldog   100       100    200
2                      3  american_pit_bull_terrier   100       100    200
3                      4               basset_hound   100       100    200
4                      5                     beagle   100       100    200
5                      6                     Bengal   100       100    200
6                      7                     Birman   100       100    200
7                      8                     Bombay    88        96    184
8                      9                      boxer    99       100    199
9                     10          British_Shorthair   100       100    200
10                    11                  chihuahua   100       100    200
11                    12               Egyptian_Mau    97        93    190

## 5. Phân bố kích thước ảnh gốc

Ảnh Oxford có kích thước gốc thay đổi. Vì vậy input resolution là một quyết định thực nghiệm, không phải hằng số data-determined như EuroSAT.


In [ ]:
size_distribution = (
    official_manifest.groupby(["width", "height"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
official_manifest["aspect_ratio"] = (
    official_manifest["width"] / official_manifest["height"]
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(official_manifest["width"], bins=30)
axes[0].set_xlabel("original width")
axes[0].set_ylabel("count")
axes[0].set_title("Original widths")
axes[1].hist(official_manifest["height"], bins=30)
axes[1].set_xlabel("original height")
axes[1].set_ylabel("count")
axes[1].set_title("Original heights")
fig.tight_layout()
size_plot_path = EXAMPLE_DIR / "oxford_original_size_distribution.png"
fig.savefig(size_plot_path, dpi=150)
plt.close(fig)

print(
    {
        "unique_size_pairs": int(len(size_distribution)),
        "min_width": int(official_manifest["width"].min()),
        "max_width": int(official_manifest["width"].max()),
        "min_height": int(official_manifest["height"].min()),
        "max_height": int(official_manifest["height"].max()),
        "figure": size_plot_path.relative_to(PROJECT_ROOT).as_posix(),
    }
)
size_distribution.head(10)

{'unique_size_pairs': 1017, 'min_width': 114, 'max_width': 3264, 'min_height': 103, 'max_height': 2606, 'figure': 'results/figures/oxford_pets/examples/oxford_original_size_distribution.png'}


     width  height  count
798    500     375   1424
756    500     333   1069
468    375     500    511
393    333     500    509
287    300     225    261
757    500     334    250
755    500     332    185
395    334     500    151
758    500     335     97
392    332     500     91

## 6. Ảnh ví dụ

Một ví dụ deterministic cho mỗi lớp được lưu để kiểm tra trực quan. File được mở và convert sang RGB trong bộ nhớ, không sửa raw image.


In [ ]:
example_rows = (
    official_manifest.sort_values("image_id")
    .groupby("class_name", sort=True)
    .head(1)
    .sort_values("class_name")
    .reset_index(drop=True)
)

columns = 6
rows = int(np.ceil(len(example_rows) / columns))
fig, axes = plt.subplots(rows, columns, figsize=(columns * 2.1, rows * 2.3))
axes = np.array(axes).reshape(-1)
for axis in axes:
    axis.axis("off")
for axis, row in zip(axes, example_rows.itertuples(index=False)):
    with Image.open(PROJECT_ROOT / row.relative_path) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(row.class_name, fontsize=7)
    axis.axis("off")
fig.tight_layout()
example_path = EXAMPLE_DIR / "oxford_class_examples.png"
fig.savefig(example_path, dpi=150)
plt.close(fig)
print(
    {
        "example_figure": example_path.relative_to(PROJECT_ROOT).as_posix(),
        "classes_shown": len(example_rows),
    }
)

{'example_figure': 'results/figures/oxford_pets/examples/oxford_class_examples.png', 'classes_shown': 37}


## 7. Biểu diễn dữ liệu

Mỗi ảnh được decode theo nội dung, convert sang RGB, resize về resolution vuông đã chọn, và biểu diễn thành tensor `float32` trong `[0, 1]`. Bốn file có tên `.jpg` nhưng nội dung PNG vẫn được giữ trong dataset nếu chúng nằm trong annotation chính thức.


## 8. Split

Official test set được giữ nguyên. Chỉ official trainval được chia thành 80% training và 20% validation bằng stratification với seed 42. Membership CSV được lưu để mọi kiến trúc dùng cùng split.


In [ ]:
train_df, val_df = train_test_split(
    official_trainval_df,
    test_size=0.20,
    random_state=SEED,
    stratify=official_trainval_df["label"],
)
test_df = official_test_df.copy()

train_df = train_df.sort_values("image_id").reset_index(drop=True)
val_df = val_df.sort_values("image_id").reset_index(drop=True)
test_df = test_df.sort_values("image_id").reset_index(drop=True)

train_df.to_csv(SPLIT_DIR / "oxford_pets_train.csv", index=False)
val_df.to_csv(SPLIT_DIR / "oxford_pets_val.csv", index=False)
test_df.to_csv(SPLIT_DIR / "oxford_pets_test.csv", index=False)

sets = [set(df["image_id"]) for df in [train_df, val_df, test_df]]
overlap_count = len(sets[0] & sets[1]) + len(sets[0] & sets[2]) + len(sets[1] & sets[2])
accounted = len(train_df) + len(val_df) + len(test_df)
assert accounted == 7349
assert overlap_count == 0
assert len(test_df) == 3669

split_distribution = (
    pd.concat(
        [
            train_df.groupby("class_name").size().rename("train"),
            val_df.groupby("class_name").size().rename("validation"),
            test_df.groupby("class_name").size().rename("test"),
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)

print(
    {
        "train": len(train_df),
        "validation": len(val_df),
        "test": len(test_df),
        "accounted": accounted,
        "overlap_count": overlap_count,
    }
)
split_distribution.head(12)

{'train': 2944, 'validation': 736, 'test': 3669, 'accounted': 7349, 'overlap_count': 0}


           class_name  train  validation  test
0          Abyssinian     80          20    98
1              Bengal     80          20   100
2              Birman     80          20   100
3              Bombay     77          19    88
4   British_Shorthair     80          20   100
5        Egyptian_Mau     74          19    97
6          Maine_Coon     80          20   100
7             Persian     80          20   100
8             Ragdoll     80          20   100
9        Russian_Blue     80          20   100
10            Siamese     79          20   100
11             Sphynx     80          20   100

## 9. Preprocessing

TensorFlow input pipeline dùng `tf.io.decode_image`, bộ giải mã phát hiện nội dung ảnh được hỗ trợ thay vì dựa vào suffix file. Resize được áp dụng sau khi convert RGB. Raw image không bị overwrite hay sửa.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def decode_resize_image(path, label, resolution):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.image.resize(image, [resolution, resolution], method="bilinear")
    image = tf.ensure_shape(image, [resolution, resolution, 3])
    return image, label


def make_dataset(df, resolution, batch_size, shuffle=False, seed=SEED):
    paths = [str(PROJECT_ROOT / path) for path in df["relative_path"].tolist()]
    labels = df["label"].astype("int32").to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(df), seed=seed, reshuffle_each_iteration=True
        )
    dataset = dataset.map(
        lambda path, label: decode_resize_image(path, label, resolution),
        num_parallel_calls=AUTOTUNE,
    )
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(AUTOTUNE)
    return dataset


sample_tensor, sample_label = decode_resize_image(
    str(PROJECT_ROOT / official_manifest.iloc[0]["relative_path"]),
    int(official_manifest.iloc[0]["label"]),
    128,
)
print(
    {
        "sample_shape_at_128": tuple(sample_tensor.shape),
        "dtype": sample_tensor.dtype.name,
        "min": float(tf.reduce_min(sample_tensor)),
        "max": float(tf.reduce_max(sample_tensor)),
        "decode": "tf.io.decode_image content-aware",
    }
)

{'sample_shape_at_128': (128, 128, 3), 'dtype': 'float32', 'min': 0.0003135531733278185, 'max': 1.0, 'decode': 'tf.io.decode_image content-aware'}


## 10. Thí nghiệm resolution

Resolution được chọn bằng evidence vì ảnh Oxford có kích thước thay đổi. BasicCNN2D được dùng cho tất cả thí nghiệm protocol. Stage A sàng lọc `96x96`, `128x128`, và `160x160` trên fixed training-only subset; Stage B xác nhận candidate mạnh hơn trên full train split nếu cần.


In [ ]:
all_results = []
HP_PATH = HP_DIR / "oxford_pets_hyperparameters.csv"


def make_stratified_subset(df, per_class, seed=SEED):
    sampled_groups = []
    for class_name, group in df.groupby("class_name", sort=True):
        sampled_groups.append(
            group.sample(n=min(per_class, len(group)), random_state=seed)
        )
    return (
        pd.concat(sampled_groups, axis=0).sort_values("image_id").reset_index(drop=True)
    )


tuning_train_df = make_stratified_subset(train_df, per_class=15, seed=SEED)
tuning_train_df.to_csv(SPLIT_DIR / "oxford_pets_tuning_train_subset.csv", index=False)


def evaluate_macro_f1(model, dataset):
    y_true_batches = []
    y_pred_batches = []
    for images, labels in dataset:
        probabilities = model.predict(images, verbose=0)
        y_true_batches.append(labels.numpy())
        y_pred_batches.append(np.argmax(probabilities, axis=1))
    return float(
        f1_score(
            np.concatenate(y_true_batches),
            np.concatenate(y_pred_batches),
            average="macro",
        )
    )


def build_training_model(
    resolution, learning_rate, use_augmentation=False, dropout_rate=0.0
):
    if dropout_rate > 0:
        inputs = keras.Input(shape=(resolution, resolution, 3), name="image")
        x = augmentation_layer(inputs) if use_augmentation else inputs
        x = keras.layers.Conv2D(32, 3, padding="same", activation="relu")(x)
        x = keras.layers.MaxPooling2D(2)(x)
        x = keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
        x = keras.layers.MaxPooling2D(2)(x)
        x = keras.layers.GlobalAveragePooling2D()(x)
        x = keras.layers.Dense(64, activation="relu")(x)
        x = keras.layers.Dropout(dropout_rate, seed=SEED)(x)
        outputs = keras.layers.Dense(37, activation="softmax")(x)
        model = keras.Model(
            inputs, outputs, name=f"basic_cnn_2d_dropout_{dropout_rate}"
        )
    else:
        base = build_basic_cnn_2d(
            input_shape=(resolution, resolution, 3), num_classes=37
        )
        inputs = keras.Input(shape=(resolution, resolution, 3), name="image")
        x = augmentation_layer(inputs) if use_augmentation else inputs
        outputs = base(x)
        model = keras.Model(inputs, outputs, name="basic_cnn_2d_training")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    return model


def run_experiment(
    experiment_type,
    candidate_value,
    tuning_stage,
    train_source,
    train_frame,
    val_frame,
    resolution,
    learning_rate,
    batch_size,
    max_epochs,
    patience,
    use_augmentation=False,
    dropout_rate=0.0,
):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    train_ds = make_dataset(
        train_frame,
        resolution=resolution,
        batch_size=batch_size,
        shuffle=True,
        seed=SEED,
    )
    val_ds = make_dataset(
        val_frame, resolution=resolution, batch_size=batch_size, shuffle=False
    )
    model = build_training_model(
        resolution=resolution,
        learning_rate=learning_rate,
        use_augmentation=use_augmentation,
        dropout_rate=dropout_rate,
    )
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience, restore_best_weights=True
        )
    ]
    start = time.perf_counter()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=max_epochs,
        callbacks=callbacks,
        verbose=0,
    )
    training_time = time.perf_counter() - start
    val_loss, val_accuracy = model.evaluate(val_ds, verbose=0)
    val_macro_f1 = evaluate_macro_f1(model, val_ds)
    epochs_run = len(history.history["loss"])
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
    result = {
        "experiment_type": experiment_type,
        "candidate_value": candidate_value,
        "tuning_stage": tuning_stage,
        "train_source": train_source,
        "resolution": resolution,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "max_epochs": max_epochs,
        "patience": patience,
        "epochs_run": epochs_run,
        "best_epoch": best_epoch,
        "augmentation": use_augmentation,
        "dropout_rate": dropout_rate,
        "train_loss_last": float(history.history["loss"][-1]),
        "train_accuracy_last": float(history.history["accuracy"][-1]),
        "val_loss": float(val_loss),
        "val_accuracy": float(val_accuracy),
        "val_macro_f1": val_macro_f1,
        "training_time_seconds": float(training_time),
        "time_per_epoch_seconds": float(training_time / epochs_run),
        "seed": SEED,
    }
    all_results.append(result)
    pd.DataFrame(all_results).to_csv(HP_PATH, index=False)
    return result, history.history


augmentation_layer = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal", seed=SEED),
        keras.layers.RandomRotation(0.05, fill_mode="reflect", seed=SEED),
        keras.layers.RandomZoom(0.10, fill_mode="reflect", seed=SEED),
        keras.layers.RandomTranslation(0.05, 0.05, fill_mode="reflect", seed=SEED),
    ],
    name="conservative_pet_photo_augmentation",
)

print(
    {
        "tuning_subset": len(tuning_train_df),
        "classes": int(tuning_train_df["class_name"].nunique()),
        "per_class_min": int(tuning_train_df.groupby("class_name").size().min()),
        "per_class_max": int(tuning_train_df.groupby("class_name").size().max()),
        "results_path": HP_PATH.relative_to(PROJECT_ROOT).as_posix(),
    }
)

{'tuning_subset': 555, 'classes': 37, 'per_class_min': 15, 'per_class_max': 15, 'results_path': 'results/hyperparameters/oxford_pets_hyperparameters.csv'}


In [ ]:
resolution_candidates = [96, 128, 160]
screening_lr = 1e-3
screening_batch_size = 32
resolution_histories = {}

for resolution in resolution_candidates:
    result, history = run_experiment(
        experiment_type="resolution",
        candidate_value=resolution,
        tuning_stage="A_subset_screening",
        train_source="stratified_train_subset_555",
        train_frame=tuning_train_df,
        val_frame=val_df,
        resolution=resolution,
        learning_rate=screening_lr,
        batch_size=screening_batch_size,
        max_epochs=3,
        patience=1,
    )
    resolution_histories[f"A_{resolution}"] = history

resolution_stage_a_df = pd.DataFrame(
    [
        row
        for row in all_results
        if row["experiment_type"] == "resolution"
        and row["tuning_stage"] == "A_subset_screening"
    ]
)
resolution_stage_a_df["relative_pixels"] = (
    resolution_stage_a_df["resolution"] ** 2
) / (96**2)
resolution_stage_a_df.sort_values(
    ["val_macro_f1", "val_loss"], ascending=[False, True]
)[
    [
        "candidate_value",
        "relative_pixels",
        "epochs_run",
        "val_loss",
        "val_accuracy",
        "val_macro_f1",
        "training_time_seconds",
    ]
]

C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adap

   candidate_value  relative_pixels  ...  val_macro_f1  training_time_seconds
0               96         1.000000  ...      0.011234              34.191898
2              160         2.777778  ...      0.008188              63.209032
1              128         1.777778  ...      0.006965              56.060060

[3 rows x 7 columns]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(
    resolution_stage_a_df["resolution"],
    resolution_stage_a_df["val_macro_f1"],
    marker="o",
)
axes[0].set_xlabel("resolution")
axes[0].set_ylabel("validation macro F1")
axes[0].set_title("Stage A resolution vs macro F1")
axes[1].plot(
    resolution_stage_a_df["resolution"],
    resolution_stage_a_df["time_per_epoch_seconds"],
    marker="o",
)
axes[1].set_xlabel("resolution")
axes[1].set_ylabel("seconds per epoch")
axes[1].set_title("Stage A resolution runtime")
fig.tight_layout()
resolution_a_plot = HP_FIG_DIR / "resolution_stage_a.png"
fig.savefig(resolution_a_plot, dpi=150)
plt.close(fig)

top_resolution_candidates = (
    resolution_stage_a_df.sort_values(
        ["val_macro_f1", "val_loss"], ascending=[False, True]
    )["resolution"]
    .head(2)
    .astype(int)
    .tolist()
)
print(
    {
        "top_resolution_candidates": top_resolution_candidates,
        "plot": resolution_a_plot.relative_to(PROJECT_ROOT).as_posix(),
    }
)

{'top_resolution_candidates': [96, 160], 'plot': 'results/figures/oxford_pets/hyperparameters/resolution_stage_a.png'}


In [ ]:
for resolution in top_resolution_candidates:
    result, history = run_experiment(
        experiment_type="resolution",
        candidate_value=resolution,
        tuning_stage="B_full_train_confirmation",
        train_source="full_train_split",
        train_frame=train_df,
        val_frame=val_df,
        resolution=resolution,
        learning_rate=screening_lr,
        batch_size=screening_batch_size,
        max_epochs=4,
        patience=1,
    )
    resolution_histories[f"B_{resolution}"] = history

resolution_stage_b_df = pd.DataFrame(
    [
        row
        for row in all_results
        if row["experiment_type"] == "resolution"
        and row["tuning_stage"] == "B_full_train_confirmation"
    ]
)
resolution_stage_b_df["relative_pixels"] = (
    resolution_stage_b_df["resolution"] ** 2
) / (96**2)
resolution_ranked = resolution_stage_b_df.sort_values(
    ["val_macro_f1", "val_loss"], ascending=[False, True]
).reset_index(drop=True)
selected_resolution = int(resolution_ranked.loc[0, "resolution"])
resolution_tradeoff_note = (
    "selected highest validation macro F1, with validation loss as tie-breaker"
)
if (
    160 in resolution_ranked["resolution"].to_list()
    and 128 in resolution_ranked["resolution"].to_list()
):
    row_160 = resolution_ranked[resolution_ranked["resolution"] == 160].iloc[0]
    row_128 = resolution_ranked[resolution_ranked["resolution"] == 128].iloc[0]
    f1_gain_160_over_128 = float(row_160["val_macro_f1"] - row_128["val_macro_f1"])
    runtime_ratio_160_over_128 = float(
        row_160["time_per_epoch_seconds"] / row_128["time_per_epoch_seconds"]
    )
    if (
        selected_resolution == 160
        and f1_gain_160_over_128 < 0.01
        and runtime_ratio_160_over_128 > 1.35
    ):
        selected_resolution = 128
        resolution_tradeoff_note = f"128 selected: 160 macro-F1 gain {f1_gain_160_over_128:.4f} with {runtime_ratio_160_over_128:.2f}x epoch time"
    else:
        resolution_tradeoff_note = f"160 vs 128 macro-F1 difference {f1_gain_160_over_128:.4f}; epoch-time ratio {runtime_ratio_160_over_128:.2f}"

print(
    {
        "selected_resolution": selected_resolution,
        "tradeoff_note": resolution_tradeoff_note,
    }
)
resolution_stage_b_df[
    [
        "candidate_value",
        "relative_pixels",
        "epochs_run",
        "val_loss",
        "val_accuracy",
        "val_macro_f1",
        "time_per_epoch_seconds",
        "training_time_seconds",
    ]
]

{'selected_resolution': 96, 'tradeoff_note': 'selected highest validation macro F1, with validation loss as tie-breaker'}


C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


   candidate_value  ...  training_time_seconds
0               96  ...             123.061002
1              160  ...             254.208236

[2 rows x 8 columns]

## 11. Augmentation

Augmentation cho ảnh thú cưng được giới hạn ở các biến đổi có khả năng giữ nguyên breed label: horizontal flip, rotation nhỏ, zoom nhỏ, và translation nhỏ. Vertical flip không được dùng vì ảnh thú cưng lộn ngược không đại diện tốt cho phân phối ảnh tự nhiên. Biến đổi màu mạnh không được thêm vì màu và pattern lông có thể liên quan đến breed.


## 12. Khảo sát learning rate

Learning rate được chọn bằng validation evidence. Stage A sàng lọc các candidate logarithmic trên fixed training-only subset, có mở rộng biên nếu candidate tốt nhất nằm ở rìa. Stage B xác nhận bằng full train split.


In [ ]:
coarse_learning_rates = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
lr_histories = {}

for learning_rate in coarse_learning_rates:
    result, history = run_experiment(
        experiment_type="learning_rate",
        candidate_value=learning_rate,
        tuning_stage="A_subset_screening",
        train_source="stratified_train_subset_555",
        train_frame=tuning_train_df,
        val_frame=val_df,
        resolution=selected_resolution,
        learning_rate=learning_rate,
        batch_size=screening_batch_size,
        max_epochs=3,
        patience=1,
    )
    lr_histories[f"A_{learning_rate}"] = history

lr_stage_a_df = pd.DataFrame(
    [
        row
        for row in all_results
        if row["experiment_type"] == "learning_rate"
        and row["tuning_stage"] == "A_subset_screening"
    ]
)
best_lr_stage_a = float(
    lr_stage_a_df.sort_values(
        ["val_macro_f1", "val_loss"], ascending=[False, True]
    ).iloc[0]["learning_rate"]
)
extension_candidates = []
if best_lr_stage_a == min(coarse_learning_rates):
    extension_candidates = [3e-5, 1e-5]
elif best_lr_stage_a == max(coarse_learning_rates):
    extension_candidates = [3e-2]

for learning_rate in extension_candidates:
    result, history = run_experiment(
        experiment_type="learning_rate",
        candidate_value=learning_rate,
        tuning_stage="A_boundary_extension",
        train_source="stratified_train_subset_555",
        train_frame=tuning_train_df,
        val_frame=val_df,
        resolution=selected_resolution,
        learning_rate=learning_rate,
        batch_size=screening_batch_size,
        max_epochs=3,
        patience=1,
    )
    lr_histories[f"A_ext_{learning_rate}"] = history

lr_screen_df = pd.DataFrame(
    [
        row
        for row in all_results
        if row["experiment_type"] == "learning_rate"
        and row["tuning_stage"].startswith("A_")
    ]
)
top_lr_candidates = (
    lr_screen_df.sort_values(["val_macro_f1", "val_loss"], ascending=[False, True])[
        "learning_rate"
    ]
    .head(2)
    .astype(float)
    .tolist()
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].semilogx(
    lr_screen_df["learning_rate"], lr_screen_df["val_macro_f1"], marker="o"
)
axes[0].set_xlabel("learning rate")
axes[0].set_ylabel("validation macro F1")
axes[0].set_title("LR screening")
axes[1].semilogx(lr_screen_df["learning_rate"], lr_screen_df["val_loss"], marker="o")
axes[1].set_xlabel("learning rate")
axes[1].set_ylabel("validation loss")
axes[1].set_title("LR validation loss")
fig.tight_layout()
lr_plot = HP_FIG_DIR / "learning_rate_stage_a.png"
fig.savefig(lr_plot, dpi=150)
plt.close(fig)

print(
    {
        "best_lr_stage_a": best_lr_stage_a,
        "extension_candidates": extension_candidates,
        "top_lr_candidates": top_lr_candidates,
        "plot": lr_plot.relative_to(PROJECT_ROOT).as_posix(),
    }
)
lr_screen_df.sort_values(["val_macro_f1", "val_loss"], ascending=[False, True])[
    [
        "candidate_value",
        "tuning_stage",
        "epochs_run",
        "val_loss",
        "val_accuracy",
        "val_macro_f1",
        "training_time_seconds",
    ]
]

{'best_lr_stage_a': 0.001, 'extension_candidates': [], 'top_lr_candidates': [0.001, 0.003], 'plot': 'results/figures/oxford_pets/hyperparameters/learning_rate_stage_a.png'}


C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adap

   candidate_value        tuning_stage  ...  val_macro_f1  training_time_seconds
2           0.0010  A_subset_screening  ...      0.011234              31.716149
3           0.0030  A_subset_screening  ...      0.004337              28.070481
4           0.0100  A_subset_screening  ...      0.001430              27.421592
1           0.0003  A_subset_screening  ...      0.001366              15.402547
0           0.0001  A_subset_screening  ...      0.001360              29.763094

[5 rows x 7 columns]

In [ ]:
for learning_rate in top_lr_candidates:
    result, history = run_experiment(
        experiment_type="learning_rate",
        candidate_value=learning_rate,
        tuning_stage="B_full_train_confirmation",
        train_source="full_train_split",
        train_frame=train_df,
        val_frame=val_df,
        resolution=selected_resolution,
        learning_rate=learning_rate,
        batch_size=screening_batch_size,
        max_epochs=5,
        patience=2,
    )
    lr_histories[f"B_{learning_rate}"] = history

lr_stage_b_df = pd.DataFrame(
    [
        row
        for row in all_results
        if row["experiment_type"] == "learning_rate"
        and row["tuning_stage"] == "B_full_train_confirmation"
    ]
)
selected_lr = float(
    lr_stage_b_df.sort_values(
        ["val_macro_f1", "val_loss"], ascending=[False, True]
    ).iloc[0]["learning_rate"]
)
print({"selected_lr": selected_lr})
lr_stage_b_df[
    [
        "candidate_value",
        "epochs_run",
        "best_epoch",
        "val_loss",
        "val_accuracy",
        "val_macro_f1",
        "training_time_seconds",
    ]
]

{'selected_lr': 0.001}


C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


   candidate_value  epochs_run  ...  val_macro_f1  training_time_seconds
0            0.001           5  ...      0.028679             164.477879
1            0.003           5  ...      0.001430              68.596963

[2 rows x 7 columns]

## 13. Khảo sát batch size

Batch size được so sánh sau khi resolution và learning rate đã cố định. Quyết định dựa trên validation macro F1 và loss, với runtime chỉ dùng như tie-breaker cuối.


In [ ]:
batch_candidates = [16, 32, 64]
batch_histories = {}
for batch_size in batch_candidates:
    result, history = run_experiment(
        experiment_type="batch_size",
        candidate_value=batch_size,
        tuning_stage="full_train_comparison",
        train_source="full_train_split",
        train_frame=train_df,
        val_frame=val_df,
        resolution=selected_resolution,
        learning_rate=selected_lr,
        batch_size=batch_size,
        max_epochs=4,
        patience=1,
    )
    batch_histories[str(batch_size)] = history

batch_df = pd.DataFrame(
    [row for row in all_results if row["experiment_type"] == "batch_size"]
)
selected_batch = int(
    batch_df.sort_values(
        ["val_macro_f1", "val_loss", "time_per_epoch_seconds"],
        ascending=[False, True, True],
    ).iloc[0]["batch_size"]
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(batch_df["batch_size"], batch_df["val_macro_f1"], marker="o")
axes[0].set_xlabel("batch size")
axes[0].set_ylabel("validation macro F1")
axes[0].set_title("Batch size vs macro F1")
axes[1].plot(batch_df["batch_size"], batch_df["time_per_epoch_seconds"], marker="o")
axes[1].set_xlabel("batch size")
axes[1].set_ylabel("seconds per epoch")
axes[1].set_title("Batch size runtime")
fig.tight_layout()
batch_plot = HP_FIG_DIR / "batch_size_comparison.png"
fig.savefig(batch_plot, dpi=150)
plt.close(fig)

print(
    {
        "selected_batch": selected_batch,
        "plot": batch_plot.relative_to(PROJECT_ROOT).as_posix(),
    }
)
batch_df[
    [
        "candidate_value",
        "epochs_run",
        "best_epoch",
        "val_loss",
        "val_accuracy",
        "val_macro_f1",
        "time_per_epoch_seconds",
        "training_time_seconds",
    ]
]

{'selected_batch': 32, 'plot': 'results/figures/oxford_pets/hyperparameters/batch_size_comparison.png'}


C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adap

   candidate_value  epochs_run  ...  time_per_epoch_seconds  training_time_seconds
0               16           4  ...                4.038167              16.152669
1               32           4  ...                4.214397              16.857586
2               64           4  ...                4.392188              17.568753

[3 rows x 8 columns]

## 14. Regularization nếu cần

Regularization không được thêm tự động. Dropout chỉ được thử khi run BasicCNN đã chọn cho thấy khoảng cách train-vs-validation accuracy lớn hơn ngưỡng kiểm tra đặt trước.


In [ ]:
baseline_match = (
    batch_df[batch_df["batch_size"] == selected_batch]
    .sort_values(["val_macro_f1", "val_loss"], ascending=[False, True])
    .iloc[0]
    .to_dict()
)
aug_result, aug_history = run_experiment(
    experiment_type="augmentation",
    candidate_value="horizontal_flip_small_rotation_zoom_translation",
    tuning_stage="full_train_comparison",
    train_source="full_train_split",
    train_frame=train_df,
    val_frame=val_df,
    resolution=selected_resolution,
    learning_rate=selected_lr,
    batch_size=selected_batch,
    max_epochs=4,
    patience=1,
    use_augmentation=True,
)
augmentation_df = pd.DataFrame([baseline_match, aug_result])
augmentation_df["candidate_value"] = [
    "none",
    "horizontal_flip_small_rotation_zoom_translation",
]
selected_augmentation = bool(
    aug_result["val_macro_f1"] > baseline_match["val_macro_f1"]
    or (
        np.isclose(aug_result["val_macro_f1"], baseline_match["val_macro_f1"])
        and aug_result["val_loss"] < baseline_match["val_loss"]
    )
)
print({"selected_augmentation": "conservative" if selected_augmentation else "none"})
augmentation_df[
    [
        "candidate_value",
        "epochs_run",
        "best_epoch",
        "val_loss",
        "val_accuracy",
        "val_macro_f1",
        "training_time_seconds",
    ]
]

{'selected_augmentation': 'none'}


C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


                                   candidate_value  ...  training_time_seconds
0                                             none  ...              16.857586
1  horizontal_flip_small_rotation_zoom_translation  ...              19.599789

[2 rows x 7 columns]

In [ ]:
selected_reference_pool = pd.DataFrame(all_results)
selected_reference_pool = selected_reference_pool[
    (selected_reference_pool["resolution"] == selected_resolution)
    & (selected_reference_pool["learning_rate"] == selected_lr)
    & (selected_reference_pool["batch_size"] == selected_batch)
]
if selected_augmentation:
    selected_reference_pool = selected_reference_pool[
        selected_reference_pool["augmentation"] == True
    ]
else:
    selected_reference_pool = selected_reference_pool[
        selected_reference_pool["augmentation"] == False
    ]
selected_reference = selected_reference_pool.sort_values(
    ["val_macro_f1", "val_loss"], ascending=[False, True]
).iloc[0]
train_val_accuracy_gap = float(
    selected_reference["train_accuracy_last"] - selected_reference["val_accuracy"]
)
regularization_note = "No dropout experiment: overfitting evidence did not exceed the preset inspection threshold."
dropout_candidates_run = []
selected_dropout = 0.0

if train_val_accuracy_gap > 0.10:
    regularization_note = (
        "Accuracy gap exceeded 0.10; dropout candidates were compared."
    )
    for dropout_rate in [0.2, 0.4]:
        result, history = run_experiment(
            experiment_type="dropout",
            candidate_value=dropout_rate,
            tuning_stage="full_train_regularization_check",
            train_source="full_train_split",
            train_frame=train_df,
            val_frame=val_df,
            resolution=selected_resolution,
            learning_rate=selected_lr,
            batch_size=selected_batch,
            max_epochs=4,
            patience=1,
            use_augmentation=selected_augmentation,
            dropout_rate=dropout_rate,
        )
        dropout_candidates_run.append(result)
    dropout_df = pd.DataFrame(dropout_candidates_run)
    best_dropout = dropout_df.sort_values(
        ["val_macro_f1", "val_loss"], ascending=[False, True]
    ).iloc[0]
    if best_dropout["val_macro_f1"] > selected_reference["val_macro_f1"]:
        selected_dropout = float(best_dropout["dropout_rate"])
        selected_reference = best_dropout
else:
    dropout_df = pd.DataFrame()

print(
    {
        "train_val_accuracy_gap": train_val_accuracy_gap,
        "regularization_note": regularization_note,
        "dropout_candidates_run": len(dropout_candidates_run),
        "selected_dropout": selected_dropout,
    }
)
(
    dropout_df
    if not dropout_df.empty
    else pd.DataFrame([{"dropout_candidates_run": 0, "reason": regularization_note}])
)

{'train_val_accuracy_gap': -0.011548914015293121, 'regularization_note': 'No dropout experiment: overfitting evidence did not exceed the preset inspection threshold.', 'dropout_candidates_run': 0, 'selected_dropout': 0.0}


   dropout_candidates_run                                             reason
0                       0  No dropout experiment: overfitting evidence di...

## 15. Giao thức huấn luyện Oxford đã chọn

Bảng dưới đây đóng băng giao thức huấn luyện cấp dataset cho Oxford để dùng ở phần so sánh kiến trúc sau. Giao thức được chọn mà không dùng official test set.


In [ ]:
results_df = pd.DataFrame(all_results)
results_df.to_csv(HP_PATH, index=False)

selected_augmentation_label = (
    "horizontal_flip_small_rotation_zoom_translation"
    if selected_augmentation
    else "none"
)
selected_regularization_label = (
    "dropout_" + str(selected_dropout) if selected_dropout > 0 else "none"
)
selected_max_epochs = int(selected_reference["max_epochs"])
selected_patience = int(selected_reference["patience"])

protocol_rows = [
    {
        "decision": "image resolution",
        "selected_value": f"{selected_resolution}x{selected_resolution} RGB",
        "classification": "experimentally selected",
        "evidence_or_reason": resolution_tradeoff_note,
    },
    {
        "decision": "augmentation",
        "selected_value": selected_augmentation_label,
        "classification": "experimentally selected",
        "evidence_or_reason": "Conservative pet-photo augmentation kept only if validation evidence exceeded no augmentation.",
    },
    {
        "decision": "optimizer/protocol choice",
        "selected_value": "Adam optimizer family; BasicCNN-only tuning; official test untouched",
        "classification": "controlled protocol choice with theoretical rationale",
        "evidence_or_reason": "One optimizer family is held fixed so architecture and selected scalar hyperparameters can be compared under a controlled protocol.",
    },
    {
        "decision": "learning rate",
        "selected_value": selected_lr,
        "classification": "experimentally selected",
        "evidence_or_reason": "Selected by Stage B full-train validation macro F1/loss after Stage A screening and boundary extension when needed.",
    },
    {
        "decision": "batch size",
        "selected_value": selected_batch,
        "classification": "experimentally selected",
        "evidence_or_reason": "Selected by validation macro F1/loss with runtime as final tie-breaker.",
    },
    {
        "decision": "regularization",
        "selected_value": selected_regularization_label,
        "classification": "experimentally selected",
        "evidence_or_reason": regularization_note,
    },
    {
        "decision": "EarlyStopping",
        "selected_value": f"monitor val_loss, patience {selected_patience}, restore_best_weights",
        "classification": "operational bound",
        "evidence_or_reason": "Limits CPU time while selecting weights from validation loss only.",
    },
    {
        "decision": "max_epochs",
        "selected_value": selected_max_epochs,
        "classification": "operational bound",
        "evidence_or_reason": "Upper bound for protocol-selection experiments; not claimed optimal.",
    },
]
protocol_df = pd.DataFrame(protocol_rows)
protocol_path = HP_DIR / "oxford_pets_selected_protocol.csv"
protocol_df.to_csv(protocol_path, index=False)

runtime_path = HP_DIR / "oxford_pets_protocol_runtime.json"
runtime_path.write_text(
    json.dumps(
        {
            "python_executable": actual_python,
            "tensorflow_version": tf.__version__,
            "experiment_rows": len(results_df),
            "recorded_training_seconds": float(
                results_df["training_time_seconds"].sum()
            ),
            "selected_resolution": selected_resolution,
            "selected_learning_rate": selected_lr,
            "selected_batch_size": selected_batch,
            "selected_augmentation": selected_augmentation_label,
            "selected_regularization": selected_regularization_label,
        },
        indent=2,
    ),
    encoding="utf-8",
)

print(
    {
        "hyperparameter_results": HP_PATH.relative_to(PROJECT_ROOT).as_posix(),
        "selected_protocol": protocol_path.relative_to(PROJECT_ROOT).as_posix(),
        "runtime_summary": runtime_path.relative_to(PROJECT_ROOT).as_posix(),
        "experiment_rows": len(results_df),
    }
)
protocol_df

{'hyperparameter_results': 'results/hyperparameters/oxford_pets_hyperparameters.csv', 'selected_protocol': 'results/hyperparameters/oxford_pets_selected_protocol.csv', 'runtime_summary': 'results/hyperparameters/oxford_pets_protocol_runtime.json', 'experiment_rows': 16}


                    decision  ...                                 evidence_or_reason
0           image resolution  ...  selected highest validation macro F1, with val...
1               augmentation  ...  Conservative pet-photo augmentation kept only ...
2  optimizer/protocol choice  ...  One optimizer family is held fixed so architec...
3              learning rate  ...  Selected by Stage B full-train validation macr...
4                 batch size  ...  Selected by validation macro F1/loss with runt...
5             regularization  ...  No dropout experiment: overfitting evidence di...
6              EarlyStopping  ...  Limits CPU time while selecting weights from v...
7                 max_epochs  ...  Upper bound for protocol-selection experiments...

[8 rows x 4 columns]

## 16. So sánh cuối cùng bốn mô hình Oxford

Giao thức huấn luyện Oxford đã chọn hiện được đóng băng. Phần này huấn luyện BasicCNN2D, AlexNetInspired2D, VGGInspired2D, và ResNetInspired2D từ đầu với cùng train split, validation split, official test split, resolution, preprocessing, augmentation policy, optimizer, learning rate, batch size, EarlyStopping, seed, và định nghĩa metric.


In [ ]:
from pathlib import Path
import json
import random
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from PIL import Image
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

from models.architectures import (
    build_alexnet_inspired_2d,
    build_basic_cnn_2d,
    build_resnet_inspired_2d,
    build_vgg_inspired_2d,
)

SEED = 42
EXPECTED_PYTHON = "C:/Users/anhca/anaconda3/envs/tf312/python.exe"
actual_python = sys.executable.replace("\\", "/")
assert actual_python.lower() == EXPECTED_PYTHON.lower(), actual_python

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "AGENTS.md").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("A05 project root not found")
    PROJECT_ROOT = PROJECT_ROOT.parent

SPLIT_DIR = PROJECT_ROOT / "results" / "splits"
HP_DIR = PROJECT_ROOT / "results" / "hyperparameters"
METRIC_DIR = PROJECT_ROOT / "results" / "metrics"
FIG_ROOT = PROJECT_ROOT / "results" / "figures" / "oxford_pets"
CURVE_DIR = FIG_ROOT / "training_curves"
CM_DIR = FIG_ROOT / "confusion_matrices"
ERROR_DIR = FIG_ROOT / "errors"
CHECKPOINT_DIR = PROJECT_ROOT / "models" / "checkpoints"
for path in [METRIC_DIR, CURVE_DIR, CM_DIR, ERROR_DIR, CHECKPOINT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(SPLIT_DIR / "oxford_pets_train.csv")
val_df = pd.read_csv(SPLIT_DIR / "oxford_pets_val.csv")
test_df = pd.read_csv(SPLIT_DIR / "oxford_pets_test.csv")
protocol_df = pd.read_csv(HP_DIR / "oxford_pets_selected_protocol.csv")
protocol = dict(zip(protocol_df["decision"], protocol_df["selected_value"]))

selected_resolution = int(str(protocol["image resolution"]).split("x")[0])
selected_lr = float(protocol["learning rate"])
selected_batch_size = int(protocol["batch size"])
selected_max_epochs = int(protocol["max_epochs"])
selected_patience = int(
    str(protocol["EarlyStopping"]).split("patience ")[1].split(",")[0]
)
selected_augmentation = str(protocol["augmentation"])
assert selected_augmentation == "none", selected_augmentation

class_info = (
    pd.concat([train_df, val_df, test_df], ignore_index=True)[["label", "class_name"]]
    .drop_duplicates()
    .sort_values("label")
)
class_names = class_info["class_name"].tolist()
label_to_class = dict(zip(class_info["label"], class_info["class_name"]))
num_classes = len(class_names)

sets = [set(df["image_id"]) for df in [train_df, val_df, test_df]]
overlap_count = len(sets[0] & sets[1]) + len(sets[0] & sets[2]) + len(sets[1] & sets[2])
assert len(train_df) == 2944 and len(val_df) == 736 and len(test_df) == 3669
assert overlap_count == 0
assert num_classes == 37

print(
    {
        "python_executable": actual_python,
        "tensorflow_version": tf.__version__,
        "devices": [str(device) for device in tf.config.list_physical_devices()],
        "train": len(train_df),
        "validation": len(val_df),
        "official_test": len(test_df),
        "classes": num_classes,
        "resolution": selected_resolution,
        "learning_rate": selected_lr,
        "batch_size": selected_batch_size,
        "max_epochs": selected_max_epochs,
        "patience": selected_patience,
    }
)

{'python_executable': 'C:/Users/anhca/anaconda3/envs/tf312/python.exe', 'tensorflow_version': '2.21.0', 'devices': ["PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')"], 'train': 2944, 'validation': 736, 'official_test': 3669, 'classes': 37, 'resolution': 96, 'learning_rate': 0.001, 'batch_size': 32, 'max_epochs': 5, 'patience': 2}


## 17. Data pipeline đã đóng băng

So sánh cuối cùng dùng các file `oxford_pets_train.csv`, `oxford_pets_val.csv`, và `oxford_pets_test.csv` đã lưu. Input pipeline giữ selected resolution, content-aware decoding, RGB conversion, và pixel scaling. Bốn file PNG-content có tên `.jpg` vẫn được include nếu thuộc official annotations.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def decode_resize_image(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.image.resize(
        image, [selected_resolution, selected_resolution], method="bilinear"
    )
    image = tf.ensure_shape(image, [selected_resolution, selected_resolution, 3])
    return image, label


def make_dataset(df, batch_size, shuffle=False, seed=SEED):
    paths = [str(PROJECT_ROOT / path) for path in df["relative_path"].tolist()]
    labels = df["label"].astype("int32").to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(df), seed=seed, reshuffle_each_iteration=True
        )
    dataset = dataset.map(decode_resize_image, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(AUTOTUNE)
    return dataset


train_eval_ds = make_dataset(train_df, selected_batch_size, shuffle=False)
val_ds = make_dataset(val_df, selected_batch_size, shuffle=False)
test_ds = make_dataset(test_df, selected_batch_size, shuffle=False)

for sample_images, sample_labels in train_eval_ds.take(1):
    sample_shape = tuple(sample_images.shape[1:])
    sample_dtype = sample_images.dtype.name

print(
    {
        "sample_shape": sample_shape,
        "dtype": sample_dtype,
        "decode": "tf.io.decode_image content-aware",
    }
)

{'sample_shape': (96, 96, 3), 'dtype': 'float32', 'decode': 'tf.io.decode_image content-aware'}


## 18. Huấn luyện bốn family kiến trúc

Mỗi kiến trúc được huấn luyện một lần dưới giao thức Oxford đã đóng băng. Validation loss chọn checkpoint qua EarlyStopping/ModelCheckpoint. Không retune siêu tham số riêng cho từng kiến trúc sau khi xem test result.


In [ ]:
MODEL_SPECS = [
    {
        "family": "BasicCNN2D",
        "key": "basic",
        "builder": build_basic_cnn_2d,
        "checkpoint": CHECKPOINT_DIR / "oxford_pets_basic.keras",
    },
    {
        "family": "AlexNetInspired2D",
        "key": "alexnet_inspired",
        "builder": build_alexnet_inspired_2d,
        "checkpoint": CHECKPOINT_DIR / "oxford_pets_alexnet_inspired.keras",
    },
    {
        "family": "VGGInspired2D",
        "key": "vgg_inspired",
        "builder": build_vgg_inspired_2d,
        "checkpoint": CHECKPOINT_DIR / "oxford_pets_vgg_inspired.keras",
    },
    {
        "family": "ResNetInspired2D",
        "key": "resnet_inspired",
        "builder": build_resnet_inspired_2d,
        "checkpoint": CHECKPOINT_DIR / "oxford_pets_resnet_inspired.keras",
    },
]


def compile_model(builder):
    model = builder(
        input_shape=(selected_resolution, selected_resolution, 3),
        num_classes=num_classes,
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=selected_lr),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    return model


def plot_history(history_df, family, key):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(history_df["epoch"], history_df["loss"], marker="o", label="train")
    axes[0].plot(
        history_df["epoch"], history_df["val_loss"], marker="o", label="validation"
    )
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("loss")
    axes[0].set_title(f"{family}: loss")
    axes[0].legend()
    axes[1].plot(history_df["epoch"], history_df["accuracy"], marker="o", label="train")
    axes[1].plot(
        history_df["epoch"], history_df["val_accuracy"], marker="o", label="validation"
    )
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("accuracy")
    axes[1].set_title(f"{family}: accuracy")
    axes[1].legend()
    fig.tight_layout()
    path = CURVE_DIR / f"{key}_history.png"
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


training_rows = []
comparison_start = time.perf_counter()

for index, spec in enumerate(MODEL_SPECS):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    train_model_ds = make_dataset(
        train_df, selected_batch_size, shuffle=True, seed=SEED
    )
    model = compile_model(spec["builder"])
    parameter_count = int(model.count_params())
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=selected_patience,
            restore_best_weights=True,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=spec["checkpoint"],
            monitor="val_loss",
            save_best_only=True,
        ),
    ]
    start = time.perf_counter()
    history = model.fit(
        train_model_ds,
        validation_data=val_ds,
        epochs=selected_max_epochs,
        callbacks=callbacks,
        verbose=0,
    )
    training_time = time.perf_counter() - start
    history_df = pd.DataFrame(history.history)
    history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
    history_path = METRIC_DIR / f"oxford_pets_{spec['key']}_history.csv"
    history_df.to_csv(history_path, index=False)
    curve_path = plot_history(history_df, spec["family"], spec["key"])
    best_index = int(history_df["val_loss"].idxmin())
    best_epoch = int(history_df.loc[best_index, "epoch"])
    early_stopping_activated = len(history_df) < selected_max_epochs
    training_rows.append(
        {
            "model_family": spec["family"],
            "key": spec["key"],
            "checkpoint_path": spec["checkpoint"].relative_to(PROJECT_ROOT).as_posix(),
            "parameter_count": parameter_count,
            "epochs_run": int(len(history_df)),
            "best_epoch": best_epoch,
            "early_stopping_activated": bool(early_stopping_activated),
            "train_loss_best_epoch_history": float(history_df.loc[best_index, "loss"]),
            "train_loss_final_epoch_history": float(history_df.iloc[-1]["loss"]),
            "train_accuracy_best_epoch_history": float(
                history_df.loc[best_index, "accuracy"]
            ),
            "train_accuracy_final_epoch_history": float(
                history_df.iloc[-1]["accuracy"]
            ),
            "validation_loss_best_epoch_history": float(
                history_df.loc[best_index, "val_loss"]
            ),
            "validation_accuracy_best_epoch_history": float(
                history_df.loc[best_index, "val_accuracy"]
            ),
            "training_time_seconds": float(training_time),
            "history_path": history_path.relative_to(PROJECT_ROOT).as_posix(),
            "history_figure": curve_path.relative_to(PROJECT_ROOT).as_posix(),
            "learning_rate": selected_lr,
            "batch_size": selected_batch_size,
            "max_epochs": selected_max_epochs,
            "patience": selected_patience,
            "seed": SEED,
        }
    )
    pd.DataFrame(training_rows).to_csv(
        METRIC_DIR / "oxford_pets_training_summaries.csv", index=False
    )
    print(
        {
            "trained": spec["family"],
            "parameters": parameter_count,
            "epochs_run": int(len(history_df)),
            "best_epoch": best_epoch,
            "early_stopping_activated": bool(early_stopping_activated),
            "training_time_seconds": round(training_time, 2),
            "completed": f"{index + 1}/{len(MODEL_SPECS)}",
        }
    )

training_summary_df = pd.DataFrame(training_rows)
training_summary_df[
    [
        "model_family",
        "parameter_count",
        "epochs_run",
        "best_epoch",
        "early_stopping_activated",
        "training_time_seconds",
    ]
]

{'trained': 'BasicCNN2D', 'parameters': 25957, 'epochs_run': 5, 'best_epoch': 5, 'early_stopping_activated': False, 'training_time_seconds': 20.75, 'completed': '1/4'}
{'trained': 'AlexNetInspired2D', 'parameters': 263653, 'epochs_run': 5, 'best_epoch': 5, 'early_stopping_activated': False, 'training_time_seconds': 36.93, 'completed': '2/4'}
{'trained': 'VGGInspired2D', 'parameters': 308293, 'epochs_run': 5, 'best_epoch': 5, 'early_stopping_activated': False, 'training_time_seconds': 70.06, 'completed': '3/4'}
{'trained': 'ResNetInspired2D', 'parameters': 327973, 'epochs_run': 5, 'best_epoch': 5, 'early_stopping_activated': False, 'training_time_seconds': 95.51, 'completed': '4/4'}


C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adap

        model_family  ...  training_time_seconds
0         BasicCNN2D  ...              20.747897
1  AlexNetInspired2D  ...              36.932364
2      VGGInspired2D  ...              70.055179
3   ResNetInspired2D  ...              95.512476

[4 rows x 6 columns]

## 19. Đánh giá official test và error artifacts

Sau khi cả bốn checkpoint đã được chọn, cell này đánh giá train, validation, và official test metrics, lưu confusion matrix 37 lớp ở kích thước đọc được, ghi per-class metrics, và tạo error grid deterministic từ các cặp nhầm lẫn test thường gặp nhất.


In [ ]:
def predict_dataset(model, dataset):
    y_true_batches = []
    y_pred_batches = []
    probability_batches = []
    start = time.perf_counter()
    for images, labels in dataset:
        probabilities = model.predict(images, verbose=0)
        probability_batches.append(probabilities)
        y_true_batches.append(labels.numpy())
        y_pred_batches.append(np.argmax(probabilities, axis=1))
    inference_time = time.perf_counter() - start
    return (
        np.concatenate(y_true_batches),
        np.concatenate(y_pred_batches),
        np.concatenate(probability_batches),
        inference_time,
    )


def split_metrics(model, dataset):
    loss, accuracy = model.evaluate(dataset, verbose=0)
    y_true, y_pred, probabilities, inference_time = predict_dataset(model, dataset)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(num_classes)),
        average="macro",
        zero_division=0,
    )
    return {
        "loss": float(loss),
        "accuracy": float(accuracy),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1),
        "y_true": y_true,
        "y_pred": y_pred,
        "probabilities": probabilities,
        "inference_time_seconds": float(inference_time),
    }


def plot_confusion_matrix(matrix, family, key):
    fig, ax = plt.subplots(figsize=(18, 16))
    im = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks(np.arange(num_classes))
    ax.set_yticks(np.arange(num_classes))
    ax.set_xticklabels(class_names, rotation=90, ha="center", fontsize=6)
    ax.set_yticklabels(class_names, fontsize=6)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(f"{family}: official test confusion matrix")
    fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    fig.tight_layout()
    path = CM_DIR / f"{key}_confusion_matrix.png"
    fig.savefig(path, dpi=180)
    plt.close(fig)
    return path


def top_confusion_pairs(matrix, top_n=12):
    rows = []
    for true_label in range(num_classes):
        for predicted_label in range(num_classes):
            if true_label == predicted_label:
                continue
            count = int(matrix[true_label, predicted_label])
            if count:
                rows.append(
                    {
                        "true_label": true_label,
                        "predicted_label": predicted_label,
                        "true_class": class_names[true_label],
                        "predicted_class": class_names[predicted_label],
                        "count": count,
                    }
                )
    return sorted(
        rows, key=lambda row: (-row["count"], row["true_class"], row["predicted_class"])
    )[:top_n]


def save_error_grid(prediction_df, pairs, family, key, examples_per_pair=4):
    selected = []
    for pair in pairs[:3]:
        pair_rows = (
            prediction_df[
                (prediction_df["true_label"] == pair["true_label"])
                & (prediction_df["predicted_label"] == pair["predicted_label"])
            ]
            .sort_values("image_id")
            .head(examples_per_pair)
        )
        selected.extend(pair_rows.to_dict(orient="records"))
    if not selected:
        return None
    columns = examples_per_pair
    rows = int(np.ceil(len(selected) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(columns * 2.4, rows * 2.7))
    axes = np.array(axes).reshape(-1)
    for axis in axes:
        axis.axis("off")
    for axis, row in zip(axes, selected):
        with Image.open(PROJECT_ROOT / row["relative_path"]) as image:
            axis.imshow(image.convert("RGB"))
        axis.set_title(
            f"T: {row['true_class']}\nP: {row['predicted_class']}", fontsize=7
        )
        axis.axis("off")
    fig.suptitle(
        f"{family}: deterministic samples from top confusion pairs", fontsize=11
    )
    fig.tight_layout()
    path = ERROR_DIR / f"{key}_top_confusion_errors.png"
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


metric_rows = []
per_class_rows = []
confusion_pair_rows = []

for spec in MODEL_SPECS:
    tf.keras.backend.clear_session()
    model = keras.models.load_model(spec["checkpoint"])
    train_metrics = split_metrics(model, train_eval_ds)
    val_metrics = split_metrics(model, val_ds)
    test_metrics = split_metrics(model, test_ds)
    matrix = confusion_matrix(
        test_metrics["y_true"], test_metrics["y_pred"], labels=list(range(num_classes))
    )
    cm_path = plot_confusion_matrix(matrix, spec["family"], spec["key"])
    pairs = top_confusion_pairs(matrix, top_n=12)
    prediction_df = test_df.copy().reset_index(drop=True)
    prediction_df["true_label"] = test_metrics["y_true"].astype(int)
    prediction_df["predicted_label"] = test_metrics["y_pred"].astype(int)
    prediction_df["true_class"] = prediction_df["true_label"].map(label_to_class)
    prediction_df["predicted_class"] = prediction_df["predicted_label"].map(
        label_to_class
    )
    prediction_df["max_probability"] = test_metrics["probabilities"].max(axis=1)
    prediction_path = METRIC_DIR / f"oxford_pets_{spec['key']}_test_predictions.csv"
    prediction_df.to_csv(prediction_path, index=False)
    error_grid_path = save_error_grid(
        prediction_df[prediction_df["true_label"] != prediction_df["predicted_label"]],
        pairs,
        spec["family"],
        spec["key"],
    )
    training_info = (
        training_summary_df[training_summary_df["key"] == spec["key"]].iloc[0].to_dict()
    )
    metric_rows.append(
        {
            "model_family": spec["family"],
            "key": spec["key"],
            "trainable_parameter_count": int(training_info["parameter_count"]),
            "epochs_run": int(training_info["epochs_run"]),
            "best_epoch": int(training_info["best_epoch"]),
            "early_stopping_activated": bool(training_info["early_stopping_activated"]),
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_precision": train_metrics["macro_precision"],
            "train_macro_recall": train_metrics["macro_recall"],
            "train_macro_f1": train_metrics["macro_f1"],
            "validation_loss": val_metrics["loss"],
            "validation_accuracy": val_metrics["accuracy"],
            "validation_macro_precision": val_metrics["macro_precision"],
            "validation_macro_recall": val_metrics["macro_recall"],
            "validation_macro_f1": val_metrics["macro_f1"],
            "test_loss": test_metrics["loss"],
            "test_accuracy": test_metrics["accuracy"],
            "test_macro_precision": test_metrics["macro_precision"],
            "test_macro_recall": test_metrics["macro_recall"],
            "test_macro_f1": test_metrics["macro_f1"],
            "training_time_seconds": float(training_info["training_time_seconds"]),
            "test_inference_time_seconds": test_metrics["inference_time_seconds"],
            "test_inference_ms_per_sample": test_metrics["inference_time_seconds"]
            / len(test_df)
            * 1000,
            "checkpoint_path": training_info["checkpoint_path"],
            "confusion_matrix_figure": cm_path.relative_to(PROJECT_ROOT).as_posix(),
            "error_grid_figure": (
                ""
                if error_grid_path is None
                else error_grid_path.relative_to(PROJECT_ROOT).as_posix()
            ),
            "prediction_path": prediction_path.relative_to(PROJECT_ROOT).as_posix(),
        }
    )
    for split_name, metrics in [
        ("train", train_metrics),
        ("validation", val_metrics),
        ("test", test_metrics),
    ]:
        per_precision, per_recall, per_f1, per_support = (
            precision_recall_fscore_support(
                metrics["y_true"],
                metrics["y_pred"],
                labels=list(range(num_classes)),
                average=None,
                zero_division=0,
            )
        )
        for class_index, class_name in enumerate(class_names):
            per_class_rows.append(
                {
                    "model_family": spec["family"],
                    "key": spec["key"],
                    "split": split_name,
                    "class_label": class_index,
                    "class_name": class_name,
                    "precision": float(per_precision[class_index]),
                    "recall": float(per_recall[class_index]),
                    "f1": float(per_f1[class_index]),
                    "support": int(per_support[class_index]),
                }
            )
    for rank, pair in enumerate(pairs, start=1):
        confusion_pair_rows.append(
            {"model_family": spec["family"], "key": spec["key"], "rank": rank, **pair}
        )
    print(
        {
            "evaluated": spec["family"],
            "test_accuracy": round(metric_rows[-1]["test_accuracy"], 4),
            "test_macro_f1": round(metric_rows[-1]["test_macro_f1"], 4),
            "test_inference_time_seconds": round(
                metric_rows[-1]["test_inference_time_seconds"], 2
            ),
        }
    )

models_df = pd.DataFrame(metric_rows)
per_class_df = pd.DataFrame(per_class_rows)
confusion_pairs_df = pd.DataFrame(confusion_pair_rows)

models_path = METRIC_DIR / "oxford_pets_models.csv"
per_class_path = METRIC_DIR / "oxford_pets_per_class_metrics.csv"
pairs_path = METRIC_DIR / "oxford_pets_confusion_pairs.csv"
models_df.to_csv(models_path, index=False)
per_class_df.to_csv(per_class_path, index=False)
confusion_pairs_df.to_csv(pairs_path, index=False)

total_comparison_runtime = time.perf_counter() - comparison_start
runtime_path = METRIC_DIR / "oxford_pets_final_runtime.json"
runtime_path.write_text(
    json.dumps(
        {
            "total_runtime_seconds": total_comparison_runtime,
            "model_training_seconds": models_df[
                ["model_family", "training_time_seconds"]
            ].to_dict(orient="records"),
            "model_test_inference_seconds": models_df[
                ["model_family", "test_inference_time_seconds"]
            ].to_dict(orient="records"),
            "python_executable": actual_python,
            "tensorflow_version": tf.__version__,
        },
        indent=2,
    ),
    encoding="utf-8",
)

models_df[
    [
        "model_family",
        "best_epoch",
        "train_loss",
        "validation_loss",
        "test_loss",
        "test_accuracy",
        "test_macro_precision",
        "test_macro_recall",
        "test_macro_f1",
        "trainable_parameter_count",
        "training_time_seconds",
        "test_inference_time_seconds",
        "early_stopping_activated",
    ]
]

{'evaluated': 'BasicCNN2D', 'test_accuracy': 0.0472, 'test_macro_f1': 0.0198, 'test_inference_time_seconds': 6.53}
{'evaluated': 'AlexNetInspired2D', 'test_accuracy': 0.0409, 'test_macro_f1': 0.0115, 'test_inference_time_seconds': 7.81}
{'evaluated': 'VGGInspired2D', 'test_accuracy': 0.0273, 'test_macro_f1': 0.0014, 'test_inference_time_seconds': 9.5}
{'evaluated': 'ResNetInspired2D', 'test_accuracy': 0.0433, 'test_macro_f1': 0.008, 'test_inference_time_seconds': 10.64}


        model_family  ...  early_stopping_activated
0         BasicCNN2D  ...                     False
1  AlexNetInspired2D  ...                     False
2      VGGInspired2D  ...                     False
3   ResNetInspired2D  ...                     False

[4 rows x 13 columns]

## 20. Bảng so sánh cuối cùng và phân tích độ phức tạp

Bảng cuối so sánh hiệu năng với parameter count, training time, và test inference time. Test score cao nhất không tự động là mô hình tốt nhất mọi trường hợp; trade-off chi phí/hiệu năng đo được là một phần của kết luận.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(
    models_df["trainable_parameter_count"], models_df["test_macro_f1"], s=80
)
for _, row in models_df.iterrows():
    axes[0].annotate(
        row["model_family"].replace("Inspired2D", ""),
        (row["trainable_parameter_count"], row["test_macro_f1"]),
        fontsize=8,
    )
axes[0].set_xscale("log")
axes[0].set_xlabel("trainable parameters (log scale)")
axes[0].set_ylabel("official test macro F1")
axes[0].set_title("Performance vs parameter count")

axes[1].scatter(models_df["training_time_seconds"], models_df["test_macro_f1"], s=80)
for _, row in models_df.iterrows():
    axes[1].annotate(
        row["model_family"].replace("Inspired2D", ""),
        (row["training_time_seconds"], row["test_macro_f1"]),
        fontsize=8,
    )
axes[1].set_xlabel("training time (seconds)")
axes[1].set_ylabel("official test macro F1")
axes[1].set_title("Performance vs training time")
fig.tight_layout()
complexity_path = FIG_ROOT / "oxford_pets_complexity_tradeoffs.png"
fig.savefig(complexity_path, dpi=150)
plt.close(fig)

display_table = models_df[
    [
        "model_family",
        "test_accuracy",
        "test_macro_f1",
        "validation_macro_f1",
        "trainable_parameter_count",
        "training_time_seconds",
        "test_inference_ms_per_sample",
        "best_epoch",
        "early_stopping_activated",
    ]
].copy()
for column in [
    "test_accuracy",
    "test_macro_f1",
    "validation_macro_f1",
    "test_inference_ms_per_sample",
]:
    display_table[column] = display_table[column].round(4)
display_table["training_time_seconds"] = display_table["training_time_seconds"].round(2)

print(
    {
        "complexity_figure": complexity_path.relative_to(PROJECT_ROOT).as_posix(),
        "total_runtime_seconds": round(total_comparison_runtime, 2),
    }
)
display_table

{'complexity_figure': 'results/figures/oxford_pets/oxford_pets_complexity_tradeoffs.png', 'total_runtime_seconds': 335.06}


        model_family  test_accuracy  ...  best_epoch  early_stopping_activated
0         BasicCNN2D         0.0472  ...           5                     False
1  AlexNetInspired2D         0.0409  ...           5                     False
2      VGGInspired2D         0.0273  ...           5                     False
3   ResNetInspired2D         0.0433  ...           5                     False

[4 rows x 9 columns]

## 21. Các cặp nhầm lẫn thường gặp nhất

Bảng sau được tính từ official test confusion matrix. Error grid lưu dưới `results/figures/oxford_pets/errors/` hiển thị các mẫu deterministic từ những cặp nhầm lẫn hàng đầu.


In [ ]:
top_pairs_table = (
    confusion_pairs_df.groupby("model_family").head(8).reset_index(drop=True)
)
top_pairs_table[["model_family", "rank", "true_class", "predicted_class", "count"]]

         model_family  rank  ...             predicted_class count
0          BasicCNN2D     1  ...  staffordshire_bull_terrier    50
1          BasicCNN2D     2  ...                Russian_Blue    38
2          BasicCNN2D     3  ...                   chihuahua    36
3          BasicCNN2D     4  ...                Russian_Blue    35
4          BasicCNN2D     5  ...                  Abyssinian    35
5          BasicCNN2D     6  ...  staffordshire_bull_terrier    35
6          BasicCNN2D     7  ...                  Abyssinian    34
7          BasicCNN2D     8  ...                  Abyssinian    33
8   AlexNetInspired2D     1  ...                Egyptian_Mau    63
9   AlexNetInspired2D     2  ...                Egyptian_Mau    59
10  AlexNetInspired2D     3  ...                Egyptian_Mau    56
11  AlexNetInspired2D     4  ...                Egyptian_Mau    54
12  AlexNetInspired2D     5  ...                Egyptian_Mau    53
13  AlexNetInspired2D     6  ...                Egyptian_Mau  

## 22. Ghi chú so sánh đo được

Với giao thức Oxford đã đóng băng, cả bốn mô hình đều hoạt động yếu trên official test split. BasicCNN2D có test metrics mạnh nhất trong bốn mô hình: accuracy `0.0472` và macro F1 `0.0198`, với 25,957 trainable parameters và 20.75 giây huấn luyện. AlexNetInspired2D, VGGInspired2D, và ResNetInspired2D tăng parameter count và runtime nhưng không tạo cải thiện có ý nghĩa dưới protocol này.

Các mô hình sâu hơn không chỉ đánh đổi compute để lấy một mức tăng accuracy nhỏ. AlexNetInspired2D đạt test accuracy `0.0409` và macro F1 `0.0115`; VGGInspired2D đạt test accuracy `0.0273` và macro F1 `0.0014`; ResNetInspired2D đạt test accuracy `0.0433` và macro F1 `0.0080`. Không mô hình nào kích hoạt EarlyStopping vì tất cả chạy hết operational bound 5 epoch đã chọn.

Các cặp nhầm lẫn thường gặp nhất của BasicCNN2D gồm Newfoundland -> staffordshire_bull_terrier, Ragdoll -> Russian_Blue, và pomeranian -> chihuahua. Error grid đã lưu cho thấy một số điểm tương đồng trực quan trong các nhóm này, ví dụ chó nhỏ nhiều lông ở pomeranian/chihuahua và mèo lông dài hoặc màu sáng ở Ragdoll/Russian_Blue. Một số nhầm lẫn thường gặp khác phản ánh khả năng tách lớp yếu hơn là một tương đồng thị giác dễ diễn giải.

Kết quả độ phức tạp vì vậy khá trực tiếp: các mô hình Oxford lớn hơn dùng nhiều tham số hơn, nhiều thời gian huấn luyện hơn, và nhiều thời gian test inference hơn nhưng không cải thiện official-test macro F1 đo được. Điều này không chứng minh kiến trúc sâu hơn luôn kém hơn cho Oxford Pets; nó chỉ cho thấy rằng dưới protocol ngắn, CPU-aware và siêu tham số được tuning bằng BasicCNN, độ sâu không đem lại lợi ích có ý nghĩa.
